# PathSnap Hackathon — Phase 2: Bayesian Tuning & HMM Route Reconstruction
### PES University — UE24CS352A: Machine Learning

**Objectives for Final Evaluation (3:00 PM):**
1. **Bayesian Hyperparameter Optimization (5 Marks):** Optimize the chosen **Random Forest** classifier using Optuna with Group-aware Cross-Validation on training trips.
2. **HMM & Viterbi Route Reconstruction (5 Marks):** Combine Random Forest candidate emission probabilities with road graph connectivity transitions to reconstruct a 100% continuous route.

## 1. Setup & Environment

In [ ]:
# Install optuna if running in Colab:
# !pip install optuna folium scikit-learn pandas numpy scipy

import os
import sys
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from scipy.spatial import KDTree

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, log_loss

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Libraries loaded successfully!")

## 2. Load Dataset & Extract Features for Tuning

In [ ]:
EARTH_RADIUS = 6371008.8

def to_metric_coords(lons, lats, ref_lon, ref_lat):
    rad_lat = np.radians(ref_lat)
    x = np.radians(lons - ref_lon) * EARTH_RADIUS * np.cos(rad_lat)
    y = np.radians(lats - ref_lat) * EARTH_RADIUS
    return x, y

def angle_difference(a, b):
    diff = abs(a - b) % 360.0
    return diff if diff <= 180.0 else 360.0 - diff

def point_to_segment_proj(px, py, ax, ay, bx, by):
    dx, dy = bx - ax, by - ay
    l2 = dx*dx + dy*dy
    if l2 == 0.0:
        return np.hypot(px - ax, py - ay), 0.0
    t = max(0.0, min(1.0, ((px - ax)*dx + (py - ay)*dy) / l2))
    proj_x = ax + t * dx
    proj_y = ay + t * dy
    return np.hypot(px - proj_x, py - proj_y), t

class TripData:
    def __init__(self, trip_id, track, nodes, arcs, route=None):
        self.trip_id = trip_id
        self.track = track
        self.nodes = nodes
        self.arcs = arcs
        self.route = route

def load_single_trip(trip_dir: Path) -> TripData:
    trip_id = trip_dir.name
    track = pd.read_csv(trip_dir / f"{trip_id}.track", sep=r"\s+", header=None, names=["lon", "lat", "t"], dtype=float)
    nodes = pd.read_csv(trip_dir / f"{trip_id}.nodes", sep=r"\s+", header=None, names=["lon", "lat"], dtype=float)
    arcs = pd.read_csv(trip_dir / f"{trip_id}.arcs", sep=r"\s+", header=None, names=["u", "v"], dtype=int)
    route_path = trip_dir / f"{trip_id}.route"
    route = pd.read_csv(route_path, sep=r"\s+", header=None, names=["arc_id"], dtype=int) if route_path.exists() else None
    return TripData(trip_id, track, nodes, arcs, route)

def extract_features_from_trip(trip: TripData, k_candidates: int = 10, is_training: bool = True) -> pd.DataFrame:
    ref_lon, ref_lat = trip.track["lon"].mean(), trip.track["lat"].mean()
    node_x, node_y = to_metric_coords(trip.nodes["lon"].values, trip.nodes["lat"].values, ref_lon, ref_lat)
    gps_x, gps_y = to_metric_coords(trip.track["lon"].values, trip.track["lat"].values, ref_lon, ref_lat)
    
    u_idx, v_idx = trip.arcs["u"].values, trip.arcs["v"].values
    ax, ay = node_x[u_idx], node_y[u_idx]
    bx, by = node_x[v_idx], node_y[v_idx]
    
    seg_lengths = np.hypot(bx - ax, by - ay)
    seg_bearings = (np.degrees(np.arctan2(bx - ax, by - ay)) + 360.0) % 360.0
    tree = KDTree(np.column_stack([(ax + bx) / 2.0, (ay + by) / 2.0]))
    
    times = trip.track["t"].values
    dt = np.diff(times, prepend=times[0] - 1.0)
    dt[dt <= 0] = 1.0
    gps_dx = np.diff(gps_x, prepend=gps_x[0])
    gps_dy = np.diff(gps_y, prepend=gps_y[0])
    speeds = np.hypot(gps_dx, gps_dy) / dt
    headings = (np.degrees(np.arctan2(gps_dx, gps_dy)) + 360.0) % 360.0
    
    true_arcs = np.full(len(gps_x), -1, dtype=int)
    if is_training and trip.route is not None:
        route_arcs = trip.route["arc_id"].values
        r_u, r_v = trip.arcs.loc[route_arcs, "u"].values, trip.arcs.loc[route_arcs, "v"].values
        r_ax, r_ay, r_bx, r_by = node_x[r_u], node_y[r_u], node_x[r_v], node_y[r_v]
        cur_idx = 0
        for i in range(len(gps_x)):
            px, py = gps_x[i], gps_y[i]
            best_d, best_r = 1e9, cur_idx
            for r in range(cur_idx, min(len(route_arcs), cur_idx + 20)):
                d, _ = point_to_segment_proj(px, py, r_ax[r], r_ay[r], r_bx[r], r_by[r])
                if d < best_d:
                    best_d, best_r = d, r
            cur_idx = best_r
            true_arcs[i] = route_arcs[best_r]
            
    _, cand_indices = tree.query(np.column_stack([gps_x, gps_y]), k=k_candidates)
    records = []
    for i in range(len(gps_x)):
        px, py = gps_x[i], gps_y[i]
        target = true_arcs[i]
        candidates = cand_indices[i]
        cand_dists = [point_to_segment_proj(px, py, ax[c], ay[c], bx[c], by[c])[0] for c in candidates]
        sorted_idx = np.argsort(cand_dists)
        for rank, s_idx in enumerate(sorted_idx):
            c = candidates[s_idx]
            d, t = point_to_segment_proj(px, py, ax[c], ay[c], bx[c], by[c])
            records.append({
                "trip_id": trip.trip_id,
                "obs_id": i,
                "candidate_arc_id": int(c),
                "dist_to_segment": float(d),
                "projection_ratio": float(t),
                "heading_diff": float(angle_difference(headings[i], seg_bearings[c])),
                "segment_length": float(seg_lengths[c]),
                "dist_to_start": float(np.hypot(px - ax[c], py - ay[c])),
                "dist_to_end": float(np.hypot(px - bx[c], py - by[c])),
                "vehicle_speed": float(speeds[i]),
                "candidate_rank": int(rank),
                "target_arc_id": int(target),
                "is_match": int(1 if (is_training and c == target) else 0)
            })
    return pd.DataFrame(records)

DATA_DIR = Path("Student folder") if Path("Student folder").exists() else Path(".")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "student_test"

def prepare_partition(folder: Path, max_trips: int = 12):
    subfolders = sorted([f for f in folder.iterdir() if f.is_dir() and not f.name.startswith(".")])[:max_trips]
    return pd.concat([extract_features_from_trip(load_single_trip(f), k_candidates=10, is_training=True) for f in subfolders], ignore_index=True)

print("Loading training data (12 trips)...", flush=True)
train_df = prepare_partition(TRAIN_DIR, max_trips=12)
print("Loading student_test data (10 trips)...", flush=True)
test_df = prepare_partition(TEST_DIR, max_trips=10)

FEATURE_COLS = ["dist_to_segment", "projection_ratio", "heading_diff", "segment_length", "dist_to_start", "dist_to_end", "vehicle_speed", "candidate_rank"]
X_train, y_train, groups_train = train_df[FEATURE_COLS].values, train_df["is_match"].values, train_df["trip_id"].values
X_test, y_test = test_df[FEATURE_COLS].values, test_df["is_match"].values
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

## 3. Bayesian Hyperparameter Optimization on Random Forest
Using Optuna (Tree-structured Parzen Estimator sampler) with Group-KFold cross validation to prevent trip leakage.

In [ ]:
def compute_top1_acc(df_eval, probs):
    df_temp = df_eval[["trip_id", "obs_id", "is_match"]].copy()
    df_temp["score"] = probs
    idx_top1 = df_temp.groupby(["trip_id", "obs_id"])["score"].idxmax()
    return float(np.mean(df_temp.loc[idx_top1, "is_match"].values))

# 1. Baseline Untuned Random Forest
print("Evaluating Untuned Baseline Random Forest...")
untuned_rf = Pipeline([("scaler", StandardScaler()), ("clf", RandomForestClassifier(n_estimators=30, max_depth=10, random_state=42, n_jobs=-1))])
untuned_rf.fit(X_train, y_train)
untuned_probs = untuned_rf.predict_proba(X_test)[:, 1]
untuned_top1 = compute_top1_acc(test_df, untuned_probs)
untuned_f1 = f1_score(y_test, (untuned_probs >= 0.5).astype(int), average="macro")
untuned_loss = log_loss(y_test, np.clip(untuned_probs, 1e-6, 1 - 1e-6))
print(f"Untuned RF -> Top-1 Acc: {untuned_top1*100:.2f}% | Test F1: {untuned_f1:.4f} | Log Loss: {untuned_loss:.4f}")

# 2. Optuna Bayesian Search Objective
sub_idx = np.arange(0, len(X_train), 3)
X_tune, y_tune, g_tune = X_train[sub_idx], y_train[sub_idx], groups_train[sub_idx]
gkf = GroupKFold(n_splits=3)

def objective(trial):
    n_est = trial.suggest_int("n_estimators", 40, 80, step=10)
    max_d = trial.suggest_int("max_depth", 10, 16)
    min_split = trial.suggest_int("min_samples_split", 5, 20)
    min_leaf = trial.suggest_int("min_samples_leaf", 2, 6)
    
    rf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=n_est, max_depth=max_d, min_samples_split=min_split, min_samples_leaf=min_leaf, random_state=42, n_jobs=-1))
    ])
    scores = [f1_score(y_tune[v], rf.fit(X_tune[tr], y_tune[tr]).predict(X_tune[v]), average="macro") for tr, v in gkf.split(X_tune, y_tune, groups=g_tune)]
    return float(np.mean(scores))

print("\nRunning 10 Bayesian Optimization Trials with Optuna...")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=10, show_progress_bar=True)

best_params = study.best_params
print(f"Best Parameters Found: {best_params}")
print(f"Best CV Score: {study.best_value:.4f}")

# 3. Refit Tuned Random Forest on full training data
tuned_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(**best_params, random_state=42, n_jobs=-1))
])
tuned_rf.fit(X_train, y_train)
tuned_probs = tuned_rf.predict_proba(X_test)[:, 1]
tuned_top1 = compute_top1_acc(test_df, tuned_probs)
tuned_f1 = f1_score(y_test, (tuned_probs >= 0.5).astype(int), average="macro")
tuned_loss = log_loss(y_test, np.clip(tuned_probs, 1e-6, 1 - 1e-6))

print(f"Tuned RF   -> Top-1 Acc: {tuned_top1*100:.2f}% | Test F1: {tuned_f1:.4f} | Log Loss: {tuned_loss:.4f}")

# Save Tuned Artifact
os.makedirs("artifacts/models", exist_ok=True)
joblib.dump(tuned_rf, "artifacts/models/random_forest_tuned.joblib")
print("Saved tuned model to artifacts/models/random_forest_tuned.joblib")

## 4. Untuned vs. Tuned Comparison Table

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model Variant": "Untuned Random Forest",
        "Configuration": "n_estimators=30, max_depth=10",
        "CV Macro F1": round(study.trials[0].value, 4),
        "Top-1 Accuracy": round(untuned_top1, 4),
        "Test Macro F1": round(untuned_f1, 4),
        "Log Loss": round(untuned_loss, 4)
    },
    {
        "Model Variant": "Tuned Random Forest (Optuna)",
        "Configuration": str(best_params),
        "CV Macro F1": round(study.best_value, 4),
        "Top-1 Accuracy": round(tuned_top1, 4),
        "Test Macro F1": round(tuned_f1, 4),
        "Log Loss": round(tuned_loss, 4)
    }
])
os.makedirs("artifacts/results", exist_ok=True)
comparison_df.to_csv("artifacts/results/bayesian_tuning_results.csv", index=False)
comparison_df

## 5. HMM & Viterbi Route Reconstruction
Comparing Classifier-only predictions vs. Classifier + HMM Viterbi decoding to prove route continuity improvement.

In [ ]:
class HMMViterbiDecoder:
    def __init__(self, arcs_df: pd.DataFrame, penalty: float = 1e-5):
        self.arc_u = arcs_df["u"].values
        self.arc_v = arcs_df["v"].values
        self.penalty = penalty

    def transition_cost(self, a_from: int, a_to: int) -> float:
        if a_from == a_to:
            return 0.85
        if self.arc_v[a_from] == self.arc_u[a_to]:
            return 0.90
        return self.penalty

    def decode(self, candidate_df: pd.DataFrame, probs: np.ndarray):
        df = candidate_df[["obs_id", "candidate_arc_id"]].copy()
        df["emission"] = np.clip(probs, 1e-6, 1.0 - 1e-6)
        obs_groups = [g for _, g in df.groupby("obs_id", sort=True)]
        if not obs_groups:
            return []
        
        viterbi_scores = np.log(obs_groups[0]["emission"].values)
        backpointers = []
        
        for t in range(1, len(obs_groups)):
            prev_cands = obs_groups[t - 1]["candidate_arc_id"].values
            curr_cands = obs_groups[t]["candidate_arc_id"].values
            curr_emissions = obs_groups[t]["emission"].values
            
            new_scores = np.zeros(len(curr_cands))
            bp = np.zeros(len(curr_cands), dtype=int)
            
            for j, c_j in enumerate(curr_cands):
                t_logs = np.log([self.transition_cost(c_i, c_j) for c_i in prev_cands])
                tot = viterbi_scores + t_logs + np.log(curr_emissions[j])
                best_i = int(np.argmax(tot))
                new_scores[j] = tot[best_i]
                bp[j] = best_i
                
            viterbi_scores = new_scores
            backpointers.append(bp)
            
        best_last = int(np.argmax(viterbi_scores))
        path = [obs_groups[-1]["candidate_arc_id"].values[best_last]]
        for t in range(len(obs_groups) - 2, -1, -1):
            best_last = backpointers[t][best_last]
            path.append(obs_groups[t]["candidate_arc_id"].values[best_last])
        path.reverse()
        return path

    @staticmethod
    def compress_route(seq):
        return [seq[i] for i in range(len(seq)) if i == 0 or seq[i] != seq[i - 1]]

    def evaluate_route_continuity(self, compressed_route):
        if len(compressed_route) <= 1: return 1.0
        conn = sum(1 for i in range(len(compressed_route) - 1) if self.arc_v[compressed_route[i]] == self.arc_u[compressed_route[i + 1]])
        return float(conn / (len(compressed_route) - 1))

# Evaluate on a sample test trip
sample_test_trip = load_single_trip(TEST_DIR / "00000000")
cand_df = extract_features_from_trip(sample_test_trip, k_candidates=10, is_training=True)
p_sample = tuned_rf.predict_proba(cand_df[FEATURE_COLS].values)[:, 1]

# 1. Classifier only Top-1
cand_df["score"] = p_sample
top1_arcs = cand_df.loc[cand_df.groupby("obs_id")["score"].idxmax(), "candidate_arc_id"].tolist()

# 2. HMM Decoder
decoder = HMMViterbiDecoder(sample_test_trip.arcs)
hmm_arcs = decoder.decode(cand_df, p_sample)

comp_top1 = decoder.compress_route(top1_arcs)
comp_hmm = decoder.compress_route(hmm_arcs)

cont_top1 = decoder.evaluate_route_continuity(comp_top1)
cont_hmm = decoder.evaluate_route_continuity(comp_hmm)

print("=== HMM DECODING IMPACT ON ROUTE CONTINUITY ===")
print(f"Classifier-Only Route Continuity: {cont_top1 * 100:.1f}%")
print(f"HMM + Viterbi Route Continuity:   {cont_hmm * 100:.1f}%")